## **Module 6: Unit Testing & CI/CD for Cloud Composer DAGs**

Because Cloud Composer continuously syncs DAG files from a Cloud Storage bucket (`gs://<composer-bucket>/dags`), uploading a Python file with a syntax error, missing import, or cyclic dependency can break DAG parsing in production.

Automated testing with `pytest` and a CI/CD pipeline (Cloud Build or GitHub Actions) prevents broken DAGs from ever reaching your Composer bucket.


### 1. Local DAG Integrity Testing with `DagBag`

Airflow provides the `DagBag` class, which loads all DAG files from a target folder and records any import or syntax errors in `dag_bag.import_errors`.


In [ ]:
# Load all DAGs in the target folder and assert zero syntax or import errors
import pytest
from airflow.models import DagBag

def test_no_import_errors():
    dag_bag = DagBag(dag_folder="04_dataproc_serverless_etl/dags", include_examples=False)
    assert not dag_bag.import_errors, f"DAG import errors: {dag_bag.import_errors}"


### 2. Enforcing Team Governance Rules

Beyond syntax checks, unit tests can enforce engineering standards across every DAG in the repository:
* Every DAG must specify at least one **tag** (`dag.tags`)
* Every DAG must configure at least **1 retry** (`default_args["retries"] >= 1`)
* Every DAG must have `catchup=False` unless explicitly exempted
* No DAG contains cyclic dependencies (`dag.check_cycle()`)

Run the workshop test suite locally using:
```bash
uv run pytest 06_testing_and_cicd/tests/test_dag_integrity.py
```


### 3. Automating Deployment to Cloud Composer via Cloud Build

Once `pytest` passes in CI, synchronize the validated DAGs to your Composer environment bucket:

```yaml
# cloudbuild.yaml snippet
steps:
  - name: "ghcr.io/astral-sh/uv:python3.11-bookworm-slim"
    entrypoint: "bash"
    args:
      - "-c"
      - |
        uv sync --frozen
        uv run pytest 06_testing_and_cicd/tests/test_dag_integrity.py

  - name: "gcr.io/cloud-builders/gsutil"
    args:
      - "-m"
      - "rsync"
      - "-d"
      - "-r"
      - "04_dataproc_serverless_etl/dags"
      - "gs://${_COMPOSER_DAG_BUCKET}/dags"
```
